# **FAERS**
# Feature Engineering, XGBoost, Counterfactual Analysis

# Load Packages

In [ ]:
import dask.dataframe as dd
import pandas as pd
from collections import Counter
import numpy as np
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
import xgboost as xgb
from sklearn.model_selection import train_test_split
import polars as pl
from sklearn.model_selection import train_test_split
from sklearn.model_selection import RandomizedSearchCV
from sklearn.utils.class_weight import compute_sample_weight
import pickle
from sklearn.metrics import accuracy_score, f1_score, log_loss, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
from dask.distributed import Client, LocalCluster
import dask_ml.model_selection as dms
import re
import dask.array as da
from xgboost import dask as dxgb
from sklearn.metrics import roc_curve, roc_auc_score

In [ ]:
from dask.distributed import Client, LocalCluster

cluster = LocalCluster(
    n_workers=1,
    threads_per_worker=4,
    processes=True
)

client = Client(cluster)

# Load Data

In [ ]:
# FAERS_final.parquet dataset
df = dd.read_parquet('FAERS_final_parquet')

# subset data
df = df.sample(frac=0.05, random_state=0).persist()

# Missing values for standard cols

In [ ]:
# these are the standard demo cols we are adding, so replacing na with -1
df['sex_bin'] = df['sex_bin'].fillna(-1)
df['wt_kg'] = df['wt_kg'].fillna(-1)
df['age_years'] = df['age_years'].fillna(-1)

# Flatten rx_name

In [ ]:
# deal with the lists of lists
def flatten_to_strings(x):
    out = []
    if x is None or (isinstance(x, float) and pd.isna(x)) or x is pd.NA:
        return out
    
    # define terms that don't count as real medical info
    exclude_terms = ["unknown", "nan", "", "no adverse event", "none", "product quality issue"]
    
    if isinstance(x, (str, int, float)):
        s = str(x).strip()
        s = re.sub(r'[\[\]<>]', '_', s)
        if s.lower() not in exclude_terms:
            out.append(s)
        return out
    
    try:
        for item in x:
            out.extend(flatten_to_strings(item))
    except TypeError:
        pass
    return list(set(out))

df['rx_flat'] = df['rx_name'].map(flatten_to_strings, meta=('rx_flat','object'))
#df['reacs_flat'] = df['reactions'].map(flatten_to_strings, meta=('reacs_flat','object'))

print("Flattened lists done")

# Top N drugs and reactions

In [ ]:
def get_strictly_flat_top(series, top_n):
    data = series.compute()
    flat_pool = [str(item) for sublist in data for item in sublist]
    counts = Counter(flat_pool)
    return [k for k, _ in counts.most_common(top_n)]

top_550_drugs = get_strictly_flat_top(df['rx_flat'], 550)
#top_300_reacs = get_strictly_flat_top(df['reacs_flat'], 300)

print("Top items calculation done")

# Unknown cols

In [ ]:
# all unknown cols
df['rx_unknown'] = df['rx_flat'].map(lambda x: 1 if len(x) == 0 else 0, meta=('rx_unknown', 'int8'))
#df['reacs_unknown'] = df['reacs_flat'].map(lambda x: 1 if len(x) == 0 else 0, meta=('reacs_unknown', 'int8'))

print("Unknown cols done")

# Target col (outcomes)

In [ ]:
import numpy as np

outcome_cols = [
    'outcome_other_serious',
    'outcome_intervention_required',
    'outcome_congenital_anomaly',
    'outcome_disability',
    'outcome_hospitalisation',
    'outcome_life_threatening',
    'outcome_death'
]

# Clean the data
df[outcome_cols] = (
    df[outcome_cols]
    .fillna(0)
    .replace('None', 0)
    .astype("int8")
)

# Define "Critical" group
critical_cols = [
    'outcome_death', 
    'outcome_life_threatening', 
    'outcome_disability', 
    'outcome_congenital_anomaly'
]

# Create the Binary Target
df["target_label"] = df[critical_cols].max(axis=1).astype("int8")

# Check new distribution
print(df["target_label"].value_counts())

# Multi-hot encoding

In [ ]:
top_rx_set = set(top_550_drugs) 
def encode_binary(lst):
    if lst is None:
        return [0] * len(top_550_drugs), 0

    s_set = set(lst)
    vec = [1 if x in s_set else 0 for x in top_550_drugs]
    other = 1 if len(s_set - top_rx_set) > 0 else 0
    return vec, other


tmp = df["rx_flat"].map(encode_binary, meta=("x", "object"))

df["rx_mh"] = tmp.map(lambda x: x[0], meta=("rx_mh", "object"))
df["other_rx"] = tmp.map(lambda x: x[1], meta=("other_rx", "int8"))

# Expand multi-hot

In [ ]:
df_indexed = df.set_index('primaryid')

def expand_rx(pdf):
    mat = pd.DataFrame(
        pdf["rx_mh"].tolist(),
        columns=top_550_drugs,
        index=pdf.index  
    ).astype("int8")
    return mat

rx_features = df_indexed.map_partitions(
    expand_rx,
    meta={n: "int8" for n in top_550_drugs}
)

rx_features = rx_features.persist()

# Final df

In [ ]:

def force_numeric(x):
    if isinstance(x, (list, np.ndarray)):
        return x[0] if len(x) > 0 else 0
    try: return float(x)
    except: return 0

print("clean and cast demo")
df_demog = df.copy()
for col in ['other_rx', 'target_label', 'age_years', 'wt_kg']:
    df_demog[col] = df_demog[col].map(force_numeric, meta=(col, 'float64'))

df_demog["primaryid"] = df_demog["primaryid"].fillna(0).astype("int64")

print("Unique pt profiles")
df_grouped = df_demog.groupby("primaryid").agg({
    'sex_bin': 'first',
    'age_years': 'first',  
    'wt_kg': 'first',
    'other_rx': 'first',
    'target_label': 'max', # If they had the reaction in ANY row, it counts as 1
})

print("Merge drug features")

df_final = dd.merge(
    df_grouped, 
    rx_features, 
    left_index=True, 
    right_index=True, 
    how="inner"
)

df_final = df_final.groupby(df_final.index).max().persist()

print("Finalize cols")
df_final = df_final.loc[:, ~df_final.columns.duplicated()]
df_final = df_final.drop(columns=['rx_mh', 'rx_flat'], errors='ignore')

df_final = df_final.reset_index()

print(f"Total Unique Patients: {len(df_final):,}")
avg_drugs = df_final[top_550_drugs].sum(axis=1).mean().compute()
print(f"Average drugs per patient: {avg_drugs:.2f}")

In [ ]:
# save engineered file
df_final_pd = df_final.compute()
df_final_pd.to_parquet("df_final.parquet", index=False)

# Train/val/test split

In [ ]:
df_final_pd = df_final.compute()
# stratified split
train_full, test_full = train_test_split(
    df_final_pd,
    test_size=0.2,
    random_state=0,
    stratify=df_final_pd["target_label"]
)

train, val = train_test_split(
    train_full,
    test_size=0.15,
    random_state=0,
    stratify=train_full["target_label"]
)

# feature/label split
X_train = train.drop(columns=["target_label"])
y_train = train["target_label"]
X_val = val.drop(columns=["target_label"])
y_val = val["target_label"]
X_test = test_full.drop(columns=["target_label"])
y_test = test_full["target_label"]

print(f"Training:   {len(X_train):,} samples | Features: {X_train.shape[1]}")
print(f"Validation: {len(X_val):,} samples")
print(f"Test:       {len(X_test):,} samples")
print("Classes in train:", sorted(y_train.unique()))

In [ ]:
search_indices = X_train.sample(frac=0.1, random_state=0).index
X_search = X_train.loc[search_indices]
y_search = y_train.loc[search_indices]

param_dist = {
    'max_depth': [6, 10, 15],
    'learning_rate': [0.05, 0.1, 0.2],
    'subsample': [0.7, 0.9],
    'colsample_bytree': [0.7, 0.9],
    'min_child_weight': [1, 2],
    'gamma': [0, 0.1, 0.3],
    'reg_lambda': [0, 0.1],
    'reg_alpha': [0, 0.1]
}

#ratio = (y_search == 0).sum() / (y_search == 1).sum()

xgb_model = xgb.XGBClassifier(
    objective="binary:logistic",
    #num_class=2,
    device="cuda",
    n_estimators=500,
    tree_method="hist",
    eval_metric="aucpr",
    scale_pos_weight = 8,
    early_stopping_rounds=50
)

search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_dist,
    n_iter=20,
    scoring='recall',
    cv=3,
    verbose=2,
    n_jobs=1
)

search.fit(
    X_search,
    y_search,
    eval_set=[(X_val, y_val)] 
)

print("Best xgboost parameters:", search.best_params_)
best_params = search.best_params_

In [ ]:
# plot aucpr curve

# Best model from search
best_model = search.best_estimator_

# Predict probabilities on validation set
y_val_probs = best_model.predict_proba(X_val)[:, 1]

# Compute precision-recall curve
precision, recall, _ = precision_recall_curve(y_val, y_val_probs)

# Compute AUC-PR
aucpr = auc(recall, precision)

# Plot
plt.figure()
plt.plot(recall, precision, label=f"AUC-PR = {aucpr:.4f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.legend()
plt.grid()

plt.show()

In [ ]:
w = compute_sample_weight("balanced", y_train)
dtrain = xgb.DMatrix(X_train, label=y_train, weight=w)
dval = xgb.DMatrix(X_val, label=y_val)

# 3. Parameters
params = {
    "objective": "binary:logistic",
    #"num_class": 2,
    "tree_method": "hist", 
    "device": "cuda",      
    **best_params          
}

xgmodel = xgb.train(
    params,
    dtrain,
    num_boost_round=500,
    evals=[(dtrain, "train"), (dval, "validation")],
    early_stopping_rounds=50,
    verbose_eval=True
)

print("Training Complete.")
print(f"Final model features: {len(xgmodel.feature_names)}")

# Save best baseline model

In [ ]:
# save model

'''best_model = search.best_estimator_
best_model.save_model("xgboost_model_final.json")'''

# To load it later
'''import xgboost as xgb
loaded_model = xgb.XGBClassifier()
loaded_model.load_model("xgboost_model_05.json")'''

# Model Evaluation

In [ ]:
# find best threshold

# Get probabilities for the validation set
probs_val = xgmodel.predict(dval)

# Calculate ROC curve points
fpr, tpr, thresholds = roc_curve(y_val, probs_val)
auc_score = roc_auc_score(y_val, probs_val)

# Find the optimal threshold 
optimal_idx = np.argmax(tpr - fpr)
optimal_threshold = thresholds[optimal_idx]

print(f"AUC Score: {auc_score:.4f}")
print(f"Best Threshold to use: {optimal_threshold:.4f}")

# Apply the new threshold
preds_val_custom = (probs_val >= optimal_threshold).astype(int)

In [ ]:
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report

xgmodel.set_param({"device": "cpu"})

dtrain_cpu = xgb.DMatrix(X_train, label=y_train)

probs_train = xgmodel.predict(dtrain_cpu)

preds_train = (probs_train >= optimal_threshold).astype(int)

print(f"Train Accuracy: {accuracy_score(y_train, preds_train):.4f}")
print("\nClassification Report (Train):")
print(classification_report(y_train, preds_train))

In [ ]:
# validation performance

# Get probabilities for the validation set
probs_val = xgmodel.predict(dval)

# Convert to binary 0/1 based on threshold
preds_val = (probs_val >= optimal_threshold).astype(int)

print(f"Validation Accuracy: {accuracy_score(y_val, preds_val):.4f}")
print("\nClassification Report:")
print(classification_report(y_val, preds_val))

In [ ]:
# test performance

# Create DMatrix for the test set
dtest = xgb.DMatrix(X_test)

# Get probability scores
probs_test = xgmodel.predict(dtest)

# Apply the optimal threshold 
preds_test = (probs_test >= optimal_threshold).astype(int)

# handle dask 
y_test_true = y_test.compute() if hasattr(y_test, 'compute') else y_test

# Metrics
print(f"Test Accuracy: {accuracy_score(y_test_true, preds_test):.4f}")
print("\nClassification Report (Precision, Recall, F1):")
print(classification_report(y_test_true, preds_test))

# Confusion Matrix Plot
cm = confusion_matrix(y_test_true, preds_test)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title(f'Test Confusion Matrix (Threshold: {optimal_threshold})')
plt.xlabel('Predicted (0=Serious, 1=Critical)')
plt.ylabel('Actual (0=Serious, 1=Critical)')
plt.show()

In [ ]:
# ex of patient prob

target_idx = 0 

try:
    prob_class_1 = probs_test[target_idx]
    prob_class_0 = 1 - prob_class_1
    
    patient_probs = [prob_class_0, prob_class_1]
    predicted_class = 1 if prob_class_1 >= optimal_threshold else 0

    print(f"Results for Patient Index: {target_idx}")

    # raw probability per class
    for class_id, score in enumerate(patient_probs):
        print(f"Class {class_id}: {score:.4f}")

    print(f"Predicted Class: {predicted_class}")
    print(f"Confidence: {patient_probs[predicted_class]:.4f}")

except IndexError:
    print(f"Error: Index {target_idx} does not exist in the dataset.")

In [ ]:
# see what features are important

importance = xgmodel.get_score(importance_type='gain')
feat_imp = pd.Series(importance).sort_values(ascending=False)

print("Top 20 Features by Gain:")
print(feat_imp.head(20))

# Counterfactual Analysis

In [ ]:
# initialize test IDs 
test_ids = X_test['primaryid'].unique()

# get the exact features the model was trained on 
model_features = xgmodel.feature_names

# build the mask for the test Pool
drug_ct = df_final[top_550_drugs].sum(axis=1)
is_critical = (df_final["target_label"] == 1)
conditions = (
    (df_final["age_years"] != -1) &
    (df_final["wt_kg"] != -1) &
    (df_final["sex_bin"] != -1)
)

mask = conditions & (drug_ct >= 2) & (drug_ct <= 10) & is_critical & (df_final['primaryid'].isin(test_ids))

# process the pool into Pandas
pool_critical_lazy = df_final.loc[mask]
pool_critical = pool_critical_lazy.compute().reset_index()
pool_critical['primaryid'] = pool_critical['primaryid'].fillna(0).astype('int64')

# predict risk scores
try:
    dpool = xgb.DMatrix(pool_critical[model_features], feature_names=model_features)
    pool_critical['risk_score'] = xgmodel.predict(dpool)
except KeyError as e:
    print(f"error - pool is missing some drug columns expected by the model: {e}")

# find 'borderline' patients near optimal threshold
window = 0.1 
borderline_pool = pool_critical[
    (pool_critical['risk_score'] >= optimal_threshold) & 
    (pool_critical['risk_score'] <= (optimal_threshold + window))
].sort_values(by='risk_score')

# print
print(f"\n test data: Found {len(borderline_pool)} Borderline Patients")
cols_to_show = ['primaryid', 'age_years', 'wt_kg', 'risk_score']
print(borderline_pool[cols_to_show].head(20))

# extract id's for counterfactual loop
pt_ids = borderline_pool['primaryid'].tolist()

In [ ]:
# pull data for borderline risk patients by primaryid

df_pd = borderline_pool

for pid in pt_ids:
    # Look inside the 'primaryid'col instead of the index
    patient_row = df_pd[df_pd['primaryid'] == pid]
    
    if not patient_row.empty:
        # Get the drugs for that specific row
        row = patient_row[top_550_drugs].iloc[0]
        
        # Find column names where value is 1
        active_rx = row[row == 1].index.tolist()
        
        print(f"Patient ID {pid}:")
        print(f"Current Drugs ({len(active_rx)}): {active_rx}\n")
    else:
        print(f"ID {pid} not found.")

In [ ]:
#features = [col for col in df_final.columns if col not in ['target_label']]

# define the scenarios for a few pts
what_if = {
    172246181: ("HYDROXYZINE", "SERTRALINE"),
    212052172: ("XANAX", "CLONAZEPAM"),
    251530591: ("FENTANYL", "OXYCODONE"),
    152407993: ("HYDROMORPHONE", "HYDROCODONE"),
    245251521: ("RISPERIDONE", "ARPIPRAZOLE"),
    204904421: ("FINASTERIDE", "MINOXIDIL"),
    188662551: ("PROPRANOLOL", "ATENONOL"),
    117376337: ("SIMVASTATIN", "PRAVASTATIN"),
    218464092: ("QUETIAPINE FUMARATE", "OLANZAPINE"),
    241421002: ("BISOPROLOL", "METOPROLOL"),
    178423923: ("MORPHINE", "OXYCODONE"),
    168049133: ("SIMVASTATIN", "LIPITOR")
}

cf_results = []

for id, (old_rx, new_rx) in what_if.items():
    # filter using the primaryid column
    pt_row = pool_critical[pool_critical['primaryid'] == id].copy()
    
    if pt_row.empty:
        print(f"Patient ID {id} not found.")
        continue

    # create DMatrix using the exact feature list the model expects
    dm_og = xgb.DMatrix(pt_row[model_features], feature_names=model_features)
    og_risk = xgmodel.predict(dm_og)[0]

    # modify
    if pt_row[old_rx].values[0] == 1:
        pt_row[old_rx] = 0 # remove old drug
        pt_row[new_rx] = 1 # add new drug 

        # create DMatrix for the modified row
        dm_new = xgb.DMatrix(pt_row[model_features], feature_names=model_features)
        new_risk = xgmodel.predict(dm_new)[0]

        # delta
        delta = new_risk - og_risk
        delta_formatted = f"{delta:+.2%}"
        
        current_threshold = optimal_threshold 

        # calculate the class shift 
        og_class = 'Critical/Permanent' if og_risk >= current_threshold else 'Serious but recoverable'
        new_class = 'Critical/Permanent' if new_risk >= current_threshold else 'Serious but recoverable'

        # store
        cf_results.append({
            "Patient ID": id,
            "Original Drug": old_rx,
            "Alternative Drug": new_rx,
            "Original risk": f"{og_risk:.2%}",
            "New Risk": f"{new_risk:.2%}",
            "Delta": delta_formatted,
            "Old Class": og_class,
            "New Class": new_class,
            "New Status": "STABLE" if new_class == 'Serious but recoverable' else "CRITICAL",
        })
    else:
        print(f"swap skipped bc patient isn't taking {old_rx}")

# output
results_df = pd.DataFrame(cf_results)
print("\n Counterfactual Results:")
print(results_df)

# For UI backend logic

In [ ]:

# save model features
import json

with open("model_features.json3", "w") as f:
    json.dump(model_features, f)

# threshold
with open("model_config_opt_threshold.json3", "w") as f:
    json.dump({"optimal_threshold": float(optimal_threshold)}, f)

# save pt dataset
pool_critical.to_parquet("pool_critical.parquet3", index=False)

In [ ]:
'''from fastapi import FastAPI
from pydantic import BaseModel
import xgboost as xgb
import json
import pandas as pd

app = FastAPI()

# load
model = xgb.Booster()
model.load_model("xgboost_model_final.json")

with open("model_features.json") as f:
    model_features = json.load(f)

with open("model_config_opt_threshold.json") as f:
    optimal_threshold = json.load(f)["optimal_threshold"]

pool_critical = pd.read_parquet("pool_critical.parquet")


# request the schema
class CounterfactualRequest(BaseModel):
    patient_id: int
    old_rx: str
    new_rx: str


# main logic
def run_counterfactual(patient_id, old_rx, new_rx):

    pt_row = pool_critical[pool_critical["primaryid"] == patient_id].copy()

    if pt_row.empty:
        return {"error": "patient not found"}

    # align features
    for col in model_features:
        if col not in pt_row.columns:
            pt_row[col] = 0

    # baseline
    dm_og = xgb.DMatrix(pt_row[model_features], feature_names=model_features)
    og_risk = float(model.predict(dm_og)[0])

    # validate drug
    if old_rx not in pt_row.columns or pt_row[old_rx].values[0] != 1:
        return {"error": "drug not present"}

    # counterfactual edit
    pt_row[old_rx] = 0

    if new_rx not in pt_row.columns:
        pt_row[new_rx] = 0
    pt_row[new_rx] = 1

    # prediction
    dm_new = xgb.DMatrix(pt_row[model_features], feature_names=model_features)
    new_risk = float(model.predict(dm_new)[0])

    return {
        "patient_id": patient_id,
        "original_risk": og_risk,
        "new_risk": new_risk,
        "delta": new_risk - og_risk,
        "old_class": "CRITICAL" if og_risk >= optimal_threshold else "SERIOUS",
        "new_class": "CRITICAL" if new_risk >= optimal_threshold else "SERIOUS"
    }


# Api 
@app.post("/counterfactual")
def counterfactual(req: CounterfactualRequest):
    return run_counterfactual(req.patient_id, req.old_rx, req.new_rx)'''